# MEC_53614 — Artificial intelligence for solids and structures — Martin Genet & Ioannis Stefanou

# L8 — Learning and acting on sequences: memory, control, agents

# E8 — Supervising a coding agent: build its harness

*Draft (2026-11-21). Session number to be confirmed (L8 in Option B of Outline v4, L7 in Option A).*

In [ ]:
# Names: …

As usual, we start with some imports…

In [ ]:
# Standard python libraries
import json
import os
import pathlib

# Computing libraries
import numpy as np    # doc: https://numpy.org/doc/stable/reference

# Plotting libraries
import matplotlib.pyplot as mpl # doc: https://matplotlib.org/stable/contents.html

# Course library (on Colab, uncomment the following line first)
# !pip install git+https://github.com/mgenet/LIB53614.git
import LIB53614 as lib

## Problem

We want the stress concentration factor $K_t$ of a finite-width plate with a central circular hole, in uniaxial tension: width $W = 50$ mm, hole diameter $d = 10$ mm, thickness $t = 1$ mm, applied stress $\sigma = 100$ MPa, linear elastic steel ($E = 210$ GPa, $\nu = 0.3$), plane stress.
$K_t = \sigma_{max} / \sigma_{net}$ is defined with respect to the nominal stress on the net section, $\sigma_{net} = \sigma W / \left(W - d\right)$.

This time, we do not write the finite element script ourselves: a coding agent writes it.
The agent is a loop around a language model (cf. `help(lib.agent)`): the model writes a script, the loop executes it, and, if something goes wrong, sends the error messages back to the model, which tries again.
The model is an open-weight model, served on the GPU server of the school: it is the same for all of you.

Our job is to make the result trustworthy—not by correcting the code ourselves, but by building the system around the agent that guides it and checks what it produces: its *harness*.
Harness engineering is the subject of the documents in `SHARED/`; its six components are AGENTS.md, rules, commands, skills, agents and hooks.

The script must respect an "interface contract", so that we can run and check it automatically: it is run as `python plate.py --W 50 --d 10 --n 24` (`--n`: mesh resolution), and its last line of output is a JSON dictionary with the keys `K_t`, `sigma_max`, `sigma_net`, `reaction_force`, `applied_force` and `n_dofs`.

In [ ]:
TASK = """Write a python script that computes, with the finite element library scikit-fem, the stress concentration factor K_t of a plate of width W with a central circular hole of diameter d, in uniaxial tension (applied stress sigma = 100 MPa at the ends, thickness 1 mm, E = 210 GPa, nu = 0.3, plane stress).
The script is run as: python plate.py --W 50 --d 10 --n 24 (lengths in mm; --n is the mesh resolution).
Its last line of output must be a JSON dictionary with the keys K_t, sigma_max, sigma_net, reaction_force, applied_force, n_dofs."""
ARGS = ["--W", 50., "--d", 10., "--n", 24]
W, d, t, sigma = 50., 10., 1., 100.

# The language model: the URL and key of the endpoint are set by the JupyterHub (environment variables).
# Without them, an offline simulator is used, which only tests the loop and the hooks: it is not a model!
if "MEC53614_LLM_URL" in os.environ:
    client = lib.agent.default_client()
    print("model:", os.environ.get("MEC53614_LLM_MODEL"))
else:
    client = lib.agent.SimulatedClient(seed=0)
    print("No model available: offline simulator.")

## A. Without a harness

**Q1. Execute the following code, which runs the agent five times on the task, without harness (the loop only sends back the error messages when the script crashes).
What are the five values of $K_t$? How spread are they?
For this geometry, Heywood's approximation gives $K_t \approx 2 + \left(1 - d/W\right)^3$ (`lib.plate.kt_heywood`): how far are the results?**

In [ ]:
runs_A = [lib.agent.run(TASK, None, client=client, args=ARGS, workdir="runs_A", name=f"A{k}") for k in range(5)]
K_t_A = [r.result["K_t"] if r.result else np.nan for r in runs_A]
print("K_t:", np.round(K_t_A, 3), f"(Heywood: {lib.plate.kt_heywood(d/W):.3f})")

**Q2. Read the scripts written by the agent (the paths are in `runs_A[k].scripts`).
List the errors you find, and, for each one, how you found it (reading, running with other arguments, comparing with a known result, a plot…).**

**Q3. The same request, sent twice, does not give the same script. Why? (Cf. the glossary in `SHARED/`, "Probabiliste".)**

## B. Under the hood: the context window

**Q4. Execute the following code, which prints everything that was sent to the model in the first run, and estimates its number of tokens (about four characters per token).
What does the model know about our project, our library, our conventions, our definition of $K_t$?
Which part of the loop decides that the task is finished?**

In [ ]:
for message in runs_A[0].messages:
    print(f"===== {message['role']} =====")
    print(message["content"][:1500] + ("…" if len(message["content"]) > 1500 else ""))
n_chars = sum(len(message["content"]) for message in runs_A[0].messages)
print(f"\nabout {n_chars // 4} tokens")

## C. Verification first: hooks

A hook is a check run by the loop (not by the model) after each script: it is deterministic.
Each hook takes the path of the script and its output (a dictionary), and returns `(ok, message)`; the message is sent to the model when the check fails.
A good hook does not need to know the answer: it checks properties that any correct result must have.

**Q5. Complete the following hooks, which check that the output contains all the required keys (with finite values), that $\sigma_{net}$ follows our definition, and the global equilibrium (the applied force on the quarter model, and the reaction on the symmetry plane).**

Hint:
* For a quarter model (by symmetry), the applied force is $\sigma t W / 2$.

In [ ]:
def check_output(path, out):
    missing = ### YOUR CODE HERE ###
    return (not missing, f"Missing or non-finite outputs: {missing}.")

def check_definition(path, out):
    expected = ### YOUR CODE HERE ###
    return (abs(out["sigma_net"] - expected) < 1e-6 * expected, f"sigma_net = {out['sigma_net']:.4g} MPa, but the nominal stress on the net section is sigma W / (W - d) = {expected:.4g} MPa.")

def check_equilibrium(path, out):
    expected = ### YOUR CODE HERE ###
    if abs(out["applied_force"] - expected) > 1e-6 * expected:
        return (False, f"applied_force = {out['applied_force']:.4g} N, but the force applied on the quarter model is sigma t W / 2 = {expected:.4g} N.")
    ok = ### YOUR CODE HERE ###
    return (ok, f"Global equilibrium violated: reaction_force = {out['reaction_force']:.6g} N, applied force = {expected:.6g} N.")

**Q6. Complete the following hook, which runs the script with two mesh resolutions, and checks that $K_t$ changes by less than 1%.
Why must it also check that the number of degrees of freedom has actually increased?**

Hint:
* `lib.agent.execute(path, args)` runs a script with a list of arguments, and returns its output (a dictionary), or an error message (a string).

In [ ]:
def check_convergence(path, out):
    out_1 = lib.agent.execute(path, ["--W", W, "--d", d, "--n", 24])
    out_2 = ### YOUR CODE HERE ###
    if isinstance(out_1, str) or isinstance(out_2, str):
        return (False, "The script fails for --n 24 or --n 48.")
    ### YOUR CODE HERE ###
    change = ### YOUR CODE HERE ###
    return (change < 0.01, f"K_t is not converged: {out_1['K_t']:.4f} (--n 24) vs. {out_2['K_t']:.4f} (--n 48), {100*change:.1f}% change.")

**Q7. For a small hole in a wide plate ($d/W \to 0$), the Kirsch solution gives $\sigma_{max} = 3 \sigma$.
What is then $K_t$ (with our definition) for $d/W = 0.02$?
Complete the following hook, which checks this limiting case.**

In [ ]:
def check_kirsch(path, out):
    out_small = ### YOUR CODE HERE ###
    if isinstance(out_small, str):
        return (False, "The script fails for a small hole (d/W = 0.02).")
    expected = ### YOUR CODE HERE ###
    return (abs(out_small["K_t"] - expected) < 0.02 * expected, f"For a small hole (d/W = 0.02), K_t = {out_small['K_t']:.3f}, whereas the Kirsch solution gives about {expected:.3f}.")

HOOKS = [check_output, check_definition, check_equilibrium, check_convergence, check_kirsch]

**Q8. Before trusting our hooks, we test them: execute the following code, which applies them to a correct script and to five scripts with typical errors (`lib.plate.VARIANTS`).
Is each faulty script caught by at least one hook, and the correct one by none?
Which hooks are the most useful? (Testing tests by introducing errors on purpose is called mutation testing.)**

In [ ]:
pathlib.Path("variants").mkdir(exist_ok=True)
for variant, description in lib.plate.VARIANTS.items():
    path = pathlib.Path("variants") / f"{variant}.py"
    path.write_text(lib.plate.script(variant))
    out = lib.agent.execute(path, ARGS)
    failed = [hook.__name__ for hook in HOOKS if not hook(path, out)[0]] if isinstance(out, dict) else ["(crash)"]
    print(f"{variant:15s} ({description}): K_t = {out['K_t']:.3f}, failed hooks: {failed}")

**Q9. One of these errors (constraining only the vertex degrees of freedom of quadratic elements on the symmetry planes) was made by an AI assistant while writing the reference solution of this exercise, and caught by the Kirsch check.
Why not simply add a hook "$K_t$ within 3% of Heywood's approximation"?**

## D. The harness

We now write the other components of the harness, in a folder `harness/` organized as in real tools: `AGENTS.md`, `rules/*.md`, `skills/*/SKILL.md` (cf. `SHARED/fiche-memo-formats.pdf`).
Here, the rules and skills are always loaded in the context window; commands and sub-agents are not used.

In [ ]:
for folder in ["harness/rules", "harness/skills/verify-fe"]:
    pathlib.Path(folder).mkdir(parents=True, exist_ok=True)

**Q10. Complete the following `AGENTS.md`: project summary, stack (with versions), architecture, commands.
Only what is useful at every request: too long, it becomes counterproductive.**

In [ ]:
### YOUR CODE HERE ###
text = """## Project summary
…
## Stack
…
## Architecture
…
## Commands
…
"""
pathlib.Path("harness/AGENTS.md").write_text(text)
print(f"harness/AGENTS.md: {len(text)} characters")

**Q11. Write at least two rules, one subject per file, each with the sections "Do", "Never" and "Expected usage" (a short example).
A rule must be checkable: no "clean" or "good". Base them on the errors you found in part A.**

In [ ]:
### YOUR CODE HERE ###
text = """---
description: …
alwaysApply: true
---
## Do
## Never
## Expected usage
"""
pathlib.Path("harness/rules/definitions.md").write_text(text)
print(f"harness/rules/definitions.md: {len(text)} characters")

In [ ]:
### YOUR CODE HERE ###
text = """---
description: …
alwaysApply: true
---
## Do
## Never
## Expected usage
"""
pathlib.Path("harness/rules/boundary_conditions.md").write_text(text)
print(f"harness/rules/boundary_conditions.md: {len(text)} characters")

**Q12. Write a skill `verify-fe`, i.e., the procedure that you would follow yourself to verify a finite element result, as ordered steps (sections Goal, Rules, Workflow).**

In [ ]:
### YOUR CODE HERE ###
text = """---
name: verify-fe
description: …
---
## Goal
## Rules
## Workflow
"""
pathlib.Path("harness/skills/verify-fe/SKILL.md").write_text(text)
print(f"harness/skills/verify-fe/SKILL.md: {len(text)} characters")

**Q13. Execute the following code, which loads the harness and prints what it adds to the context window.
For each error found in Q2, which component of the harness addresses it, and why that one? (Cf. "Diagnostic" in the glossary: the agent forgets an information → AGENTS.md; conventions not respected → a rule; a complete know-how is needed → a skill; it must be guaranteed → a hook.)**

In [ ]:
harness = lib.agent.Harness.from_folder("harness", hooks=HOOKS)
print(harness.context())
print(f"\nabout {len(harness.context()) // 4} tokens")

## E. With the harness

**Q14. Execute the following code, which runs the agent five times with the harness.
Compare with part A: values of $K_t$ and their spread, distance to Heywood, number of calls to the model, number of tokens, duration.
Which errors remain?**

In [ ]:
runs_E = [lib.agent.run(TASK, harness, client=client, args=ARGS, workdir="runs_E", name=f"E{k}") for k in range(5)]
K_t_E = [r.result["K_t"] if (r.result and r.success) else np.nan for r in runs_E]
print("K_t:", np.round(K_t_E, 3), f"(Heywood: {lib.plate.kt_heywood(d/W):.3f})")
for label, runs in [("without harness", runs_A), ("with harness", runs_E)]:
    print(f"{label:16s}: calls = {np.mean([r.n_calls for r in runs]):.1f}, tokens = {np.mean([r.n_tokens for r in runs]):.0f}, duration = {np.mean([r.duration for r in runs]):.0f} s")

mpl.plot(np.zeros(5), K_t_A, "o", label="without harness")
mpl.plot(np.ones(5), K_t_E, "o", label="with harness")
mpl.axhline(lib.plate.kt_heywood(d/W), color="k", ls="--", label="Heywood")
mpl.xticks([0, 1], ["without", "with"]); mpl.ylabel("K_t"); mpl.legend(); mpl.grid();

**Q15. Complete and execute the following code, which runs the agent with the hooks only (no AGENTS.md, rules or skills), and with the text components only (no hooks).
Which components matter most here? Why is a rule not a guarantee, while a hook is?**

In [ ]:
harness_hooks_only = ### YOUR CODE HERE ###
harness_text_only = ### YOUR CODE HERE ###
for label, h in [("hooks only", harness_hooks_only), ("text only", harness_text_only)]:
    runs = [lib.agent.run(TASK, h, client=client, args=ARGS, workdir="runs_ablation", name=f"{label.replace(' ', '_')}{k}", verbose=False) for k in range(5)]
    K_t = [r.result["K_t"] if r.result else np.nan for r in runs]
    n_ok = sum(all(hook(r.scripts[-1], r.result)[0] for hook in HOOKS) if r.result else False for r in runs) # checked afterwards, with all the hooks
    print(f"{label:10s}: K_t = {np.round(K_t, 3)}, {n_ok}/5 pass all the checks, calls = {np.mean([r.n_calls for r in runs]):.1f}")

## F. Verification is not validation

**Q16. All the hooks pass. Is $K_t$ "right"?
What do the hooks not check (think about the modeling assumptions: plane stress, linear elasticity, perfect geometry, finite length of the plate…)?
Relate to the difference between verification ("solving the equations right") and validation ("solving the right equations").**

**Q17. Who is responsible for the value of $K_t$ that would be used to size a part: the model, the tool, the harness, or you?
In one paragraph, compare supervising an agent with supervising a junior engineer.**

## Bonus

Use the same harness with a real coding tool on your own machine (e.g., an open-source agent connected to the same endpoint).
`AGENTS.md` is a convention shared by several tools; rules, skills and hooks have tool-specific formats (cf. `SHARED/fiche-memo-formats.pdf`).
How do your hooks translate (e.g., a test command run after each edit)?

## AI audit

Please indicate in the following cell (the format of this audit will be detailed soon):
* which AI tools you used, besides the agent of this exercise, and for what;
* every error you found in their outputs, and how you found it;
* what you would ask, or forbid, an assistant next time to avoid these errors.

In [ ]:
# AI audit: …